# 🚀 Full OBB Training Pipeline

This notebook:
1. Mounts Google Drive (data persists across disconnections)
2. Generates 5000 hardcore synthetic QR images with OBB labels
3. Trains YOLOv8-OBB model
4. Saves `best.pt` to your Drive

**Run all cells and wait ~1-2 hours (depending on GPU).**

In [ ]:
# ============== 1. MOUNT GOOGLE DRIVE ==============
from google.colab import drive
drive.mount('/content/drive')

# Create working directory on Drive
import os
DRIVE_DIR = '/content/drive/MyDrive/qr_obb_training'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Working directory: {DRIVE_DIR}')

In [ ]:
# ============== 2. INSTALL DEPENDENCIES ==============
!pip install ultralytics qrcode pillow -q

import ultralytics
ultralytics.checks()

In [ ]:
# ============== 3. DATASET GENERATOR (HARDCORE) ==============

import os
import random
import cv2
import numpy as np
import qrcode
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance
from io import BytesIO
import yaml

# ===== CONFIGURATION =====
OUTPUT_DIR = f"{DRIVE_DIR}/dataset"
NUM_IMAGES = 5000
TRAIN_RATIO = 0.8
IMG_SIZE = 640
QR_SIZE_RANGE = (60, 450)

AUG_PROBS = {
    'jpeg_artifacts': 0.7,
    'gaussian_blur': 0.4,
    'motion_blur': 0.3,
    'gaussian_noise': 0.5,
    'brightness': 0.6,
    'contrast': 0.5,
    'shadow': 0.4,
    'perspective': 0.5,
}

def ensure_dirs():
    for split in ['train', 'val']:
        os.makedirs(f'{OUTPUT_DIR}/images/{split}', exist_ok=True)
        os.makedirs(f'{OUTPUT_DIR}/labels/{split}', exist_ok=True)

def generate_random_qr_content():
    types = ['url', 'text', 'json', 'email', 'phone']
    t = random.choice(types)
    if t == 'url':
        return f'https://example.com/{random.randint(1000, 99999)}'
    elif t == 'email':
        return f'user{random.randint(1000,9999)}@mail.com'
    elif t == 'json':
        return f'{{"id": {random.randint(1,1000)}, "val": {random.randint(100,9999)}}}'
    elif t == 'phone':
        return f'+7{random.randint(9000000000, 9999999999)}'
    else:
        return f'Data-{random.randint(100000, 999999)}'

def create_qr_image():
    content = generate_random_qr_content()
    version = random.randint(1, 8)
    error_correction = random.choice([
        qrcode.constants.ERROR_CORRECT_L,
        qrcode.constants.ERROR_CORRECT_M,
        qrcode.constants.ERROR_CORRECT_Q,
        qrcode.constants.ERROR_CORRECT_H,
    ])
    box_size = random.randint(6, 14)
    border = random.randint(2, 6)
    
    if random.random() < 0.15:
        fill_color = (random.randint(0, 60), random.randint(0, 60), random.randint(0, 60))
        back_color = (random.randint(200, 255), random.randint(200, 255), random.randint(200, 255))
    else:
        fill_color = 'black'
        back_color = 'white'
    
    qr = qrcode.QRCode(version=version, error_correction=error_correction, box_size=box_size, border=border)
    qr.add_data(content)
    qr.make(fit=True)
    qr_img = qr.make_image(fill_color=fill_color, back_color=back_color).convert('RGBA')
    
    target_size = random.randint(QR_SIZE_RANGE[0], QR_SIZE_RANGE[1])
    qr_img = qr_img.resize((target_size, target_size), Image.LANCZOS)
    return qr_img

def rotate_qr(qr_img, angle):
    w, h = qr_img.size
    canvas = Image.new('RGBA', (w * 2, h * 2), (255, 255, 255, 0))
    canvas.paste(qr_img, (w // 2, h // 2))
    rotated = canvas.rotate(angle, expand=True, resample=Image.BICUBIC)
    
    half_w, half_h = w / 2, h / 2
    corners = [(-half_w, -half_h), (half_w, -half_h), (half_w, half_h), (-half_w, half_h)]
    rad = -np.radians(angle)
    cos_a, sin_a = np.cos(rad), np.sin(rad)
    rot_corners = [(x * cos_a - y * sin_a, x * sin_a + y * cos_a) for x, y in corners]
    rcx, rcy = rotated.width / 2, rotated.height / 2
    final_corners = [(rcx + rx, rcy + ry) for rx, ry in rot_corners]
    return rotated, final_corners

def create_background():
    bg_type = random.choice(['solid', 'gradient', 'noise'])
    if bg_type == 'solid':
        color = random.randint(180, 255)
        return Image.new('RGB', (IMG_SIZE, IMG_SIZE), (color, color, color))
    elif bg_type == 'gradient':
        arr = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        c1 = np.array([random.randint(150, 255) for _ in range(3)])
        c2 = np.array([random.randint(150, 255) for _ in range(3)])
        for y in range(IMG_SIZE):
            t = y / IMG_SIZE
            arr[y, :] = (c1 * (1 - t) + c2 * t).astype(np.uint8)
        return Image.fromarray(arr)
    else:
        noise = np.random.randint(180, 255, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        return Image.fromarray(noise)

def apply_jpeg_artifacts(img):
    quality = random.randint(15, 60)
    buffer = BytesIO()
    img.save(buffer, format='JPEG', quality=quality)
    buffer.seek(0)
    return Image.open(buffer).convert('RGB')

def apply_gaussian_blur(img):
    return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 3.0)))

def apply_motion_blur(img):
    arr = np.array(img)
    size = random.choice([5, 7, 9, 11])
    kernel = np.zeros((size, size))
    kernel[size // 2, :] = 1 / size
    angle = random.randint(0, 180)
    M = cv2.getRotationMatrix2D((size / 2, size / 2), angle, 1)
    kernel = cv2.warpAffine(kernel, M, (size, size))
    kernel = kernel / (kernel.sum() + 1e-6)
    blurred = cv2.filter2D(arr, -1, kernel)
    return Image.fromarray(blurred)

def apply_gaussian_noise(img):
    arr = np.array(img).astype(np.float32)
    noise = np.random.normal(0, random.uniform(5, 30), arr.shape)
    return Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))

def apply_brightness(img):
    return ImageEnhance.Brightness(img).enhance(random.uniform(0.4, 1.6))

def apply_contrast(img):
    return ImageEnhance.Contrast(img).enhance(random.uniform(0.5, 1.5))

def apply_shadow(img):
    arr = np.array(img).astype(np.float32)
    h, w = arr.shape[:2]
    x1, y1 = random.randint(0, w//2), random.randint(0, h//2)
    x2, y2 = random.randint(w//2, w), random.randint(h//2, h)
    arr[y1:y2, x1:x2] *= random.uniform(0.3, 0.7)
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def apply_perspective(img, corners):
    arr = np.array(img)
    h, w = arr.shape[:2]
    strength = random.uniform(0.05, 0.2)
    src_pts = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    dst_pts = src_pts.copy()
    for i in range(4):
        dst_pts[i, 0] += random.uniform(-strength * w, strength * w)
        dst_pts[i, 1] += random.uniform(-strength * h, strength * h)
    M = cv2.getPerspectiveTransform(src_pts, dst_pts)
    warped = cv2.warpPerspective(arr, M, (w, h), borderValue=(200, 200, 200))
    new_corners = []
    for cx, cy in corners:
        pt = np.array([[[cx, cy]]], dtype=np.float32)
        transformed = cv2.perspectiveTransform(pt, M)
        new_corners.append((transformed[0, 0, 0], transformed[0, 0, 1]))
    return Image.fromarray(warped), new_corners

def apply_augmentations(img, corners):
    if random.random() < AUG_PROBS['perspective']:
        img, corners = apply_perspective(img, corners)
    if random.random() < AUG_PROBS['gaussian_blur']:
        img = apply_gaussian_blur(img)
    if random.random() < AUG_PROBS['motion_blur']:
        img = apply_motion_blur(img)
    if random.random() < AUG_PROBS['gaussian_noise']:
        img = apply_gaussian_noise(img)
    if random.random() < AUG_PROBS['brightness']:
        img = apply_brightness(img)
    if random.random() < AUG_PROBS['contrast']:
        img = apply_contrast(img)
    if random.random() < AUG_PROBS['shadow']:
        img = apply_shadow(img)
    if random.random() < AUG_PROBS['jpeg_artifacts']:
        img = apply_jpeg_artifacts(img)
    return img, corners

def generate_sample(index, split):
    qr_img = create_qr_image()
    angle = random.uniform(0, 360)
    qr_rotated, corners = rotate_qr(qr_img, angle)
    bg = create_background()
    
    qr_w, qr_h = qr_rotated.size
    max_x, max_y = IMG_SIZE - qr_w, IMG_SIZE - qr_h
    
    if max_x < 0 or max_y < 0:
        scale = min(IMG_SIZE / qr_w, IMG_SIZE / qr_h) * 0.9
        new_size = (int(qr_w * scale), int(qr_h * scale))
        qr_rotated = qr_rotated.resize(new_size, Image.LANCZOS)
        corners = [(x * scale, y * scale) for x, y in corners]
        qr_w, qr_h = qr_rotated.size
        max_x, max_y = IMG_SIZE - qr_w, IMG_SIZE - qr_h
    
    paste_x = random.randint(0, max(0, max_x))
    paste_y = random.randint(0, max(0, max_y))
    bg.paste(qr_rotated, (paste_x, paste_y), qr_rotated)
    global_corners = [(x + paste_x, y + paste_y) for x, y in corners]
    bg, global_corners = apply_augmentations(bg, global_corners)
    
    filename = f'qr_{index:05d}.jpg'
    bg.save(f'{OUTPUT_DIR}/images/{split}/{filename}', quality=90)
    
    with open(f'{OUTPUT_DIR}/labels/{split}/{filename.replace(".jpg", ".txt")}', 'w') as f:
        norm = []
        for x, y in global_corners:
            norm.extend([f'{min(max(x/IMG_SIZE,0),1):.6f}', f'{min(max(y/IMG_SIZE,0),1):.6f}'])
        f.write(f"0 {' '.join(norm)}\n")

print('Generator loaded. Running generation...')

In [ ]:
# ============== 4. GENERATE DATASET ==============
ensure_dirs()
print(f'Generating {NUM_IMAGES} images to {OUTPUT_DIR}...')

for i in range(NUM_IMAGES):
    split = 'train' if i < NUM_IMAGES * TRAIN_RATIO else 'val'
    generate_sample(i, split)
    if i % 500 == 0:
        print(f'Generated {i}/{NUM_IMAGES}...')

print('Dataset generation complete!')

In [ ]:
# ============== 5. CREATE data.yaml ==============
data_config = {
    'path': OUTPUT_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'qr_code'}
}

yaml_path = f'{DRIVE_DIR}/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print(f'Created {yaml_path}')

In [ ]:
# ============== 6. TRAIN YOLO-OBB ==============
from ultralytics import YOLO

model = YOLO('yolov8n-obb.pt')  # OBB base model

results = model.train(
    data=yaml_path,
    epochs=30,
    imgsz=640,
    batch=16,
    project=DRIVE_DIR,
    name='training_run',
    exist_ok=True,
    patience=10,
    save=True,
    plots=True,
)

In [ ]:
# ============== 7. COPY BEST MODEL ==============
import shutil

best_model_src = f'{DRIVE_DIR}/training_run/weights/best.pt'
best_model_dst = f'{DRIVE_DIR}/best_obb_final.pt'

if os.path.exists(best_model_src):
    shutil.copy(best_model_src, best_model_dst)
    print(f'✅ Model saved to: {best_model_dst}')
else:
    print(f'❌ Model not found at {best_model_src}')
    print('Check training_run folder manually.')

In [ ]:
# ============== 8. QUICK TEST ==============
if os.path.exists(best_model_dst):
    test_model = YOLO(best_model_dst)
    
    # Test on a val image
    val_images = os.listdir(f'{OUTPUT_DIR}/images/val')
    if val_images:
        test_img = f'{OUTPUT_DIR}/images/val/{val_images[0]}'
        results = test_model.predict(test_img, save=True, project=DRIVE_DIR, name='test_pred')
        print(f'Test prediction saved to {DRIVE_DIR}/test_pred/')
else:
    print('No model to test.')

## ✅ Done!

Your trained model is at:
```
Google Drive > MyDrive > qr_obb_training > best_obb_final.pt
```

Download it and use in your application!